In [ ]:
import os                       # access to environment variables 
from dotenv import load_dotenv  # reads your .env file
from openai import OpenAI       # the OpenAI SDK
import gradio as gr             # the UI library

load_dotenv()                   # finds .env in your folder and loads OPENAI_API_KEY into the environment
client = OpenAI()               # creates the client — automatically reads OPENAI_API_KEY from the environment

In [ ]:
SYSTEM_PROMPT = """
You are Kingston, a patient and encouraging Python teacher for absolute beginners.
You have 10 years of experience teaching people who have never coded before.

Guidelines:
- Use simple, everyday language — no jargon without explanation.
- Always explain the "why" behind code, not just the "what".
- Break everything down into small, easy steps.
- Use short, relatable examples (cooking, sports, everyday life) to explain concepts.
- When a student makes a mistake, never make them feel bad — celebrate that they tried.
- If a question is unclear, ask one simple clarifying question before answering.
- End each explanation with an encouraging line and a small challenge to try next.
- Never write more than 10 lines of code at once — keep it digestible.
"""

In [19]:
def chat(message: str, history: list) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]
    
    messages.extend(history)
    
    messages.append({
        "role": "user",
        "content": message
    })
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=500,
        temperature=0.7
    )
    
    return response.choices[0].message.content

In [20]:
with gr.Blocks() as demo:
    
    gr.Markdown("## Python Tutor Chatbot")
    
    chatbot = gr.Chatbot(height=400)
    
    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask me anything about Python!",
            show_label=False,
            scale=8
        )
        send_btn = gr.Button("Send", scale=2, variant="stop")
        
    gr.ClearButton([msg, chatbot], value="Clear Chat")
    
    gr.Examples(
        examples=[
            "What is a variable in Python?",
            "How do I write a for loop?",
            "Can you explain what a function is with an example?",
            "I'm getting a syntax error, what does that mean?"
        ],
        inputs=msg,
    )
    
    def respond(message, history):
        reply = chat(message, history)
    
        history.append({
        "role": "user",
        "content": message
        })
    
        history.append({
        "role": "assistant",
        "content": reply
        })
    
        return "", history

    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    send_btn.click(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])

demo.launch()
        

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
